# GPT‑2 + психологически обоснованная reward-функция для KION

Этот ноутбук дообучает **вашу исходную GPT‑2**, которая предсказывает следующий выбранный фильм. Все остальные компоненты обучаются с нуля:

1. **Choice model** оценивает относительную вероятность выбора фильма среди кандидатов.
2. **Completion model** оценивает ожидаемую долю просмотра при условии, что фильм уже выбран.
3. **Support/OOD model** штрафует кандидатов, для которых в исторических данных мало поддержки.
4. GPT‑2 дообучается policy-gradient методом с KL-якорем и behavior-cloning guardrail.

Итоговая item-часть награды:

\[
R_i=\alpha\log P_{choice,i}+\beta E[watched\_pct_i]
    +\gamma novelty_i-\lambda repeat_i-\mu OOD_i.
\]

Награда slate — ожидаемое значение `R_i` под choice-моделью плюс
`δ × intra-list diversity`.

Научная основа:

- [Non-compensatory decision making for movie choice](https://cau.scholarworks.kr/item/54c55e7e-0a00-474d-b777-3ee64717b837): выбор моделируется как отсев, затем финальное решение.
- [The Influence of Online Ratings on Film Choice](https://doi.org/10.15581/003.32.2.45-59): социальные сигналы и популярность снижают неопределённость выбора.
- [Computationally modeling mood management theory](https://doi.org/10.1093/joc/jqad020): аффективный выбор стохастичен; нельзя жёстко кодировать правило «настроение → жанр».
- [Popularity-bias survey](https://doi.org/10.1007/s11257-024-09406-0): популярность полезна как сигнал, но опасна как самоподдерживающаяся петля.
- [Top-K off-policy correction](https://research.google/pubs/top-k-off-policy-correction-for-a-reinforce-recommender-system/) и [offline RL evaluation](https://arxiv.org/abs/2301.00993): исторические логи не являются несмещённой обратной связью.

**Честное ограничение KION.** Здесь нет impression-логов: мы не знаем, что пользователю показывали и что он отверг. Поэтому choice model обучается на наблюдавшемся выборе и сложных псевдоотрицательных кандидатах исходной GPT‑2. Это полезный re-ranker/reward proxy, но не причинная вероятность выбора. `watched_pct` также известен только для выбранного фильма. OOD-штраф и KL-якорь уменьшают, но не устраняют этот риск.

> **Версия для репозитория.** Встроенные `base64/zlib`-блоки удалены. Общий feature-store находится в `src/kion_reward_model.py`, а компактный внешний metadata-файл описан в `artifacts/manifest.json`.


## 1. Установка

Запустите один раз в новом Google Colab. Если Colab попросит перезапуск среды, перезапустите её и продолжите со следующей ячейки.


In [ ]:
%pip install -q "torch>=2.6,<3" "transformers==4.57.1" "catboost>=1.2.8,<1.3" "duckdb>=1.4,<2" "pandas>=2.2,<4" "numpy>=1.26,<3" "pyarrow>=17" "scikit-learn>=1.5,<2" "tqdm>=4.66,<5" "safetensors>=0.4.5" matplotlib


## 2. Пути и параметры

Нужны `prepared.zip` из предыдущего шага и checkpoint исходной GPT‑2. Поддерживаются папка `save_pretrained`, `.pt/.pth` и `.safetensors`.

Значения по умолчанию рассчитаны на бесплатный Colab. Для первого проверочного запуска можно поставить `AUX_TRAIN_CONTEXTS=10_000`, `RL_TRAIN_CONTEXTS=5_000`, `RL_EPOCHS=1`.


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DATA_PATH = Path('/content/drive/MyDrive/T-lab/dataset/prepared.zip')
CHECKPOINT_PATH = Path('/content/drive/MyDrive/T-lab/gpt2_checkpoint')
MODEL_CONFIG_JSON = None       # задайте Path(...), только если .pt не соответствует resolved_config.json
STATE_DICT_KEY = None          # например, 'model_state_dict'; обычно определяется автоматически
RUN_NAME = 'psych_reward_rl_v1'
OUTPUT_DIR = Path('/content/drive/MyDrive/T-lab') / RUN_NAME
WORK_DIR = Path('/content') / RUN_NAME

SEED = 42
AUX_TRAIN_CONTEXTS = 60_000    # choice + completion train
AUX_VALID_CONTEXTS = 15_000    # поздняя часть train; не официальный validation
CHOICE_NEGATIVES = 5
REFERENCE_POOL = 64
CONTEXT_BATCH_SIZE = 64

RL_TRAIN_CONTEXTS = 30_000
RL_EPOCHS = 2
RL_BATCH_SIZE = 24
RL_LEARNING_RATE = 2e-5
RL_TEMPERATURE = 0.9
SLATE_SIZE = 10
POLICY_POOL = 64
KL_COEF = 0.08
BC_COEF = 0.20
MAX_GRAD_NORM = 1.0

EVAL_USERS = 5_000
EVAL_BATCH_SIZE = 64
ALLOWED_RELATIVE_NDCG_DROP = 0.05
OPEN_TEST = False

REWARD_WEIGHTS = {
    'alpha_log_choice': 0.15,
    'beta_watched_pct': 0.50,
    'gamma_novelty': 0.15,
    'delta_diversity': 0.10,
    'lambda_repeat': 0.20,
    'mu_ood': 0.25,
}


## 3. Импорты, воспроизводимость и безопасная подготовка папок

Существующая папка результата не перезаписывается. Из ZIP извлекаются только ожидаемые файлы; код из архива не выполняется.


In [ ]:
import os, sys, json, math, time, random, shutil, zipfile, hashlib, platform, gc, copy
from collections import defaultdict
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import GPT2Config, GPT2LMHeadModel
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from sklearn.metrics import roc_auc_score, log_loss, mean_absolute_error, mean_squared_error
from IPython.display import display

if OUTPUT_DIR.exists():
    raise FileExistsError(f'{OUTPUT_DIR} уже существует. Измените RUN_NAME, чтобы не перезаписать опыт.')
OUTPUT_DIR.mkdir(parents=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
print('Устройство:', DEVICE)

def dump_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''): h.update(block)
    return h.hexdigest()


## 4. Данные и аудит доступного контекста

Ноутбук автоматически проверяет, есть ли в `last_watch_dt` время суток. В исходном подготовленном KION обычно сохранена только календарная дата; в этом случае часовые признаки отключаются, а не имитируются.


In [ ]:
REQUIRED = [
    'train_interactions.csv', 'validation_interactions.csv', 'test_interactions.csv',
    'validation_examples.npz', 'test_examples.npz', 'item_mapping.json',
    'resolved_config.json', 'split_boundaries.json', 'input_sha256.json'
]
DATA_DIR = WORK_DIR / 'prepared'
DATA_DIR.mkdir(exist_ok=True)

if DATA_PATH.is_dir():
    source_dir = DATA_PATH
elif DATA_PATH.is_file() and DATA_PATH.suffix.lower() == '.zip':
    with zipfile.ZipFile(DATA_PATH) as zf:
        names = {Path(n).name: n for n in zf.namelist() if not n.endswith('/')}
        for name in REQUIRED:
            if name not in names: raise FileNotFoundError(f'В архиве нет {name}')
            target = DATA_DIR / name
            with zf.open(names[name]) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)
    source_dir = DATA_DIR
else:
    raise FileNotFoundError(DATA_PATH)

for name in REQUIRED:
    src = source_dir / name
    if not src.is_file(): raise FileNotFoundError(src)
    if src.parent != DATA_DIR:
        shutil.copy2(src, DATA_DIR / name)

mapping = json.loads((DATA_DIR/'item_mapping.json').read_text(encoding='utf-8'))
cfg = json.loads((DATA_DIR/'resolved_config.json').read_text(encoding='utf-8'))
boundaries = json.loads((DATA_DIR/'split_boundaries.json').read_text(encoding='utf-8'))
item_to_index = {str(k): int(v) for k, v in mapping['item_id_to_index'].items()}
index_to_item = np.asarray(mapping['index_to_item_id'], dtype=np.int64)
N_ITEMS = len(index_to_item)
PAD = int(mapping['pad_token_id'])
MAX_CONTEXT = int(cfg['data']['training_sequences']['max_length'])

usecols = ['user_id','item_id','last_watch_dt','watched_pct','source_row_index','item_index']
dtypes = {'user_id':'int64','item_id':'int64','watched_pct':'float32',
          'source_row_index':'int64','item_index':'int32'}
interactions = {}
raw_has_time = False
for split in ['train','validation','test']:
    p = DATA_DIR/f'{split}_interactions.csv'
    probe = pd.read_csv(p, usecols=['last_watch_dt'], nrows=20_000)
    raw_has_time |= probe.last_watch_dt.astype(str).str.contains(r'[T ]\d{2}:', regex=True).any()
    frame = pd.read_csv(p, usecols=usecols, dtype=dtypes)
    frame['event_time'] = pd.to_datetime(frame.pop('last_watch_dt'), errors='raise')
    frame['split'] = split
    frame = frame.sort_values(['user_id','event_time','source_row_index'], kind='stable').reset_index(drop=True)
    interactions[split] = frame
    print(split, f'{len(frame):,}', frame.event_time.min(), frame.event_time.max())

TIME_OF_DAY_AVAILABLE = bool(raw_has_time and any(
    (f.event_time.dt.hour.ne(0) | f.event_time.dt.minute.ne(0)).any() for f in interactions.values()))
context_audit = {
    'time_of_day_available': TIME_OF_DAY_AVAILABLE,
    'session_duration_available': False,
    'reason': ('sub-day timestamps detected' if TIME_OF_DAY_AVAILABLE else
               'last_watch_dt has calendar dates only; hour and true session duration are not identifiable'),
    'duplicate_user_item_pairs': int(sum(f.duplicated(['user_id','item_id']).sum() for f in interactions.values())),
}
dump_json(OUTPUT_DIR/'context_signal_audit.json', context_audit)
display(pd.DataFrame([context_audit]))


## 5. Метаданные и исторические признаки без утечки будущего

В ноутбук встроены компактные характеристики фильмов из того же снимка KION: тип контента, основной жанр/страна, год, возрастной рейтинг и число жанров. Исторические средние пользователя, фильма и жанра строятся строгим условием `history_date < cutoff`.


In [ ]:
# Общий feature-store и метаданные вынесены из ноутбука в отдельные файлы.
def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'kion_reward_model.py').is_file():
            return candidate
    raise FileNotFoundError(
        'Не найден src/kion_reward_model.py. Запускайте ноутбук из клонированного репозитория.'
    )

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
metadata_candidates = [
    PROJECT_ROOT / 'artifacts' / 'kion_reward_item_features.csv',
    Path('/content/drive/MyDrive/T-lab/reward_model_v1/kion_reward_item_features.csv'),
]
METADATA_PATH = next((path for path in metadata_candidates if path.is_file()), None)
if METADATA_PATH is None:
    raise FileNotFoundError(
        'Не найден kion_reward_item_features.csv. Скопируйте его из reward_model_v1 '
        'в artifacts/ или укажите путь в этой ячейке.'
    )

sys.path.insert(0, str(SRC_DIR))
import kion_reward_model as rm

print('Feature-store module:', SRC_DIR / 'kion_reward_model.py')
print('Catalog metadata:', METADATA_PATH)

meta = pd.read_csv(METADATA_PATH)
LIGHT_GENRES = {'комедии','семейное','мультфильм','мультфильмы','мультсериалы','детские',
                'для детей','развлекательные','юмор','стендап','мюзиклы','музыкальные','караоке'}
SERIOUS_GENRES = {'драмы','документальное','исторические','историческое','военные','биография',
                  'научно-популярные','криминал','детективы'}
meta['item_light'] = meta.main_genre.fillna('no_genre').isin(LIGHT_GENRES).astype('float32')
meta['item_serious'] = meta.main_genre.fillna('no_genre').isin(SERIOUS_GENRES).astype('float32')
meta['item_index'] = meta.item_id.astype(str).map(item_to_index).fillna(-1).astype('int32')
meta_by_index = meta.set_index('item_index', drop=False)
genre_by_index = np.full(N_ITEMS, 'no_genre', dtype=object)
light_by_index = np.zeros(N_ITEMS, dtype=np.float32)
serious_by_index = np.zeros(N_ITEMS, dtype=np.float32)
ok = meta.item_index.between(0, N_ITEMS-1)
genre_by_index[meta.loc[ok,'item_index']] = meta.loc[ok,'main_genre'].fillna('no_genre').astype(str)
light_by_index[meta.loc[ok,'item_index']] = meta.loc[ok,'item_light']
serious_by_index[meta.loc[ok,'item_index']] = meta.loc[ok,'item_serious']

db_path = WORK_DIR/'feature_store.duckdb'
if db_path.exists(): db_path.unlink()
store = rm.FeatureStore(db_path, memory_limit='3GB', threads=4)
store.ingest(DATA_DIR, METADATA_PATH, mapping)
store.build_histories(boundaries['test']['start'])
print('Feature store готов:', db_path)


## 6. Загрузка исходной GPT‑2 и замораживание reference policy

Reference-модель никогда не обновляется. Её оценки входят в choice model и reward, поэтому обучаемая политика не может искусственно повысить reward простым увеличением собственных logits — это защита от одного из видов reward hacking.


In [ ]:
def load_checkpoint(path):
    path = Path(path)
    if not path.exists(): raise FileNotFoundError(path)
    if path.is_dir():
        model, info = GPT2LMHeadModel.from_pretrained(
            str(path), local_files_only=True, attn_implementation='eager', output_loading_info=True)
        problems = {k:v for k,v in info.items() if v}
        if problems: raise ValueError(f'Checkpoint загружен не полностью: {problems}')
    else:
        if MODEL_CONFIG_JSON is not None:
            model_cfg = GPT2Config.from_json_file(str(MODEL_CONFIG_JSON))
        else:
            valid_keys = set(GPT2Config().to_dict())
            kwargs = {k:v for k,v in cfg['model'].items() if k in valid_keys}
            model_cfg = GPT2Config(**kwargs)
        model_cfg._attn_implementation = 'eager'
        model = GPT2LMHeadModel(model_cfg)
        if path.suffix == '.safetensors':
            from safetensors.torch import load_file
            state = load_file(str(path), device='cpu')
        else:
            state = torch.load(path, map_location='cpu', weights_only=True)
        if STATE_DICT_KEY is not None:
            state = state[STATE_DICT_KEY]
        elif isinstance(state, dict):
            for key in ['model_state_dict','state_dict','model']:
                if key in state and isinstance(state[key], dict):
                    state = state[key]; break
        if not isinstance(state, dict) or not all(isinstance(v,torch.Tensor) for v in state.values()):
            raise ValueError('Не найден state_dict')
        for prefix in ['module.','_orig_mod.']:
            if state and all(k.startswith(prefix) for k in state):
                state = {k[len(prefix):]:v for k,v in state.items()}
        if model.config.tie_word_embeddings:
            a,b='transformer.wte.weight','lm_head.weight'
            if a in state and b not in state: state[b]=state[a]
            if b in state and a not in state: state[a]=state[b]
        model.load_state_dict(state, strict=True)
    if model.config.vocab_size != N_ITEMS+1 or model.config.pad_token_id != PAD:
        raise ValueError('Словарь checkpoint не совпадает с prepared.zip')
    if model.config.n_positions < MAX_CONTEXT:
        raise ValueError('n_positions меньше длины истории')
    nearby = (path if path.is_dir() else path.parent)/'item_mapping.json'
    if nearby.exists():
        other = json.loads(nearby.read_text(encoding='utf-8'))
        if other['item_id_to_index'] != mapping['item_id_to_index']:
            raise ValueError('Mapping рядом с весами не совпадает с prepared')
    else:
        print('Предупреждение: mapping рядом с весами отсутствует; проверена только размерность словаря.')
    return model.float().to(DEVICE)

reference_model = load_checkpoint(CHECKPOINT_PATH).eval()
for p in reference_model.parameters(): p.requires_grad_(False)
policy_model = copy.deepcopy(reference_model).train()
for p in policy_model.parameters(): p.requires_grad_(True)
print(f'Параметров GPT-2: {sum(p.numel() for p in policy_model.parameters()):,}')


## 7. Психологические прокси и контексты обучения

Для каждого target используются только события предыдущих календарных дней. Прокси:

- день недели;
- логарифм паузы после последнего просмотра;
- доли недосмотров `<50%` и досмотров `≥80%` среди пяти последних событий;
- частота переключения основного жанра;
- доля лёгкого контента и последний `watched_pct`;
- длина известной истории.

Они описывают состояние, но **не объявляются истинным настроением** пользователя.


In [ ]:
train_df = interactions['train']
train_counts = np.bincount(train_df.loc[train_df.item_index.ge(0),'item_index'], minlength=N_ITEMS).astype(np.int64)
novelty_by_index = -np.log((train_counts + 1) / (train_counts.sum() + N_ITEMS))
novelty_by_index = (novelty_by_index / novelty_by_index.max()).astype(np.float32)

def state_from_prior(prior, cutoff, history):
    cutoff = pd.Timestamp(cutoff)
    recent = prior.tail(5)
    y = pd.to_numeric(recent.watched_pct, errors='coerce').to_numpy(dtype=float)
    y = y[np.isfinite(y)] / 100.0
    idx = recent.item_index.to_numpy(dtype=int)
    idx = idx[(idx >= 0) & (idx < N_ITEMS)]
    genres = genre_by_index[idx]
    switches = float(np.mean(genres[1:] != genres[:-1])) if len(genres) > 1 else 0.0
    last_day = prior.event_time.max() if len(prior) else pd.NaT
    gap = float((cutoff.normalize() - last_day.normalize()).days) if pd.notna(last_day) else 9999.0
    hour = float(cutoff.hour) if TIME_OF_DAY_AVAILABLE else 0.0
    return {
        'dow_sin': float(np.sin(2*np.pi*cutoff.dayofweek/7)),
        'dow_cos': float(np.cos(2*np.pi*cutoff.dayofweek/7)),
        'hour_sin': float(np.sin(2*np.pi*hour/24)) if TIME_OF_DAY_AVAILABLE else 0.0,
        'hour_cos': float(np.cos(2*np.pi*hour/24)) if TIME_OF_DAY_AVAILABLE else 0.0,
        'time_available': float(TIME_OF_DAY_AVAILABLE),
        'gap_days_log': float(np.log1p(max(0.0, min(gap, 9999.0)))),
        'recent_incomplete': float(np.mean(y < .5)) if len(y) else .5,
        'recent_complete': float(np.mean(y >= .8)) if len(y) else .5,
        'recent_genre_switch': switches,
        'recent_light': float(light_by_index[idx].mean()) if len(idx) else 0.0,
        'last_watch_pct': float(y[-1]) if len(y) else .5,
        'history_len_log': float(np.log1p(len(history))),
    }

STATE_COLUMNS = ['dow_sin','dow_cos','hour_sin','hour_cos','time_available','gap_days_log',
                 'recent_incomplete','recent_complete','recent_genre_switch','recent_light',
                 'last_watch_pct','history_len_log']

def sample_contexts(frame, n, start=None, end=None, seed=42):
    f = frame
    users = f.user_id.to_numpy()
    items = f.item_index.to_numpy()
    dates = f.event_time.to_numpy(dtype='datetime64[ns]')
    known = items >= 0
    previous_same_user = np.r_[False, users[1:] == users[:-1]]
    eligible = known & previous_same_user
    if start is not None: eligible &= dates >= np.datetime64(start)
    if end is not None: eligible &= dates <= np.datetime64(end)
    positions = np.flatnonzero(eligible)
    rng = np.random.default_rng(seed)
    rng.shuffle(positions)
    is_new_user = np.r_[True, users[1:] != users[:-1]]
    group_starts = np.maximum.accumulate(np.where(is_new_user, np.arange(len(f)), 0))
    records=[]
    for pos in tqdm(positions, desc='Контексты', leave=False):
        gs = int(group_starts[pos]); cutoff = pd.Timestamp(dates[pos])
        # Строго предыдущие календарные дни: внутри дня порядок KION не содержит времени.
        block = f.iloc[gs:pos]
        prior = block[block.event_time < cutoff.normalize()]
        hist = prior.loc[prior.item_index.ge(0),'item_index'].to_numpy(dtype=np.int32)[-MAX_CONTEXT:]
        if len(hist) == 0: continue
        state = state_from_prior(prior, cutoff, hist)
        rec = {'context_id':len(records),'user_id':int(users[pos]),
               'target_index':int(items[pos]),'target_item_id':int(f.item_id.iloc[pos]),
               'cutoff':cutoff.normalize(),'history':hist,
               'watched':float(f.watched_pct.iloc[pos]/100.0) if np.isfinite(f.watched_pct.iloc[pos]) else np.nan}
        rec.update(state); records.append(rec)
        if len(records) >= n: break
    if len(records) < n: print(f'Получено {len(records):,} контекстов вместо {n:,}')
    return pd.DataFrame(records)

train_end = pd.Timestamp(boundaries['train']['end'])
valid_start = train_end - pd.Timedelta(days=9)
aux_train = sample_contexts(train_df, AUX_TRAIN_CONTEXTS, end=valid_start-pd.Timedelta(days=1), seed=SEED)
aux_valid = sample_contexts(train_df, AUX_VALID_CONTEXTS, start=valid_start, end=train_end, seed=SEED+1)
assert set(aux_train.context_id).isdisjoint(set())
print('Aux train/valid:', len(aux_train), len(aux_valid), 'граница:', valid_start.date())


## 8. Кандидаты исходной GPT‑2

Для каждого наблюдавшегося target берём сложные отрицательные примеры из Top‑64 исходной GPT‑2. Так choice model учится различать реалистичные конкурирующие фильмы, а не очевидно случайные ID.


In [ ]:
def collate_context_rows(rows):
    batch = list(rows)
    width = max(len(x) for x in batch)
    ids = np.full((len(batch), width), PAD, dtype=np.int64)
    mask = np.zeros((len(batch), width), dtype=np.int64)
    for i,x in enumerate(batch):
        ids[i,:len(x)] = x; mask[i,:len(x)] = 1
    return torch.as_tensor(ids,device=DEVICE), torch.as_tensor(mask,device=DEVICE)

@torch.inference_mode()
def reference_top_candidates(contexts, pool=64):
    all_idx=[]; all_scores=[]; target_scores=[]
    reference_model.eval()
    for start in tqdm(range(0,len(contexts),CONTEXT_BATCH_SIZE), desc='Reference GPT'):
        part=contexts.iloc[start:start+CONTEXT_BATCH_SIZE]
        ids,mask=collate_context_rows(part.history.tolist())
        out=reference_model(input_ids=ids,attention_mask=mask,use_cache=False).logits
        last=mask.sum(1)-1
        scores=out[torch.arange(len(part),device=DEVICE),last,:N_ITEMS].float()
        top=torch.topk(scores,k=pool,dim=1,largest=True,sorted=True)
        target=torch.as_tensor(part.target_index.to_numpy(),device=DEVICE)
        all_idx.append(top.indices.cpu().numpy().astype(np.int32))
        all_scores.append(top.values.cpu().numpy().astype(np.float32))
        target_scores.append(scores.gather(1,target[:,None]).squeeze(1).cpu().numpy().astype(np.float32))
    return np.vstack(all_idx),np.vstack(all_scores),np.concatenate(target_scores)

def build_choice_pairs(contexts, top_idx, top_score, target_score):
    rows=[]
    for r,(ci,cs,ts) in enumerate(zip(top_idx,top_score,target_score)):
        ctx=contexts.iloc[r]; target=int(ctx.target_index)
        target_pos=np.flatnonzero(ci==target)
        target_rank=int(target_pos[0]+1) if len(target_pos) else REFERENCE_POOL+1
        selected=[(target,float(ts),target_rank,1)]
        for rank,(item,score) in enumerate(zip(ci,cs),1):
            item=int(item)
            if item==target or any(item==x[0] for x in selected): continue
            selected.append((item,float(score),rank,0))
            if len(selected)==CHOICE_NEGATIVES+1: break
        for item,score,rank,label in selected:
            rows.append({'context_id':int(ctx.context_id),'user_id':int(ctx.user_id),
                         'item_index':item,'item_id':int(index_to_item[item]),
                         'cutoff':ctx.cutoff,'event_day':ctx.cutoff,'y':np.nan,
                         'gpt_ref_logit':score,'gpt_ref_rank':rank,
                         'label':label})
    pairs=pd.DataFrame(rows)
    g=pairs.groupby('context_id').gpt_ref_logit
    pairs['gpt_ref_logit_z']=((pairs.gpt_ref_logit-g.transform('mean'))/
                              g.transform('std').fillna(1).clip(lower=1e-4)).astype('float32')
    pairs['gpt_ref_rank_norm']=(pairs.gpt_ref_rank/(REFERENCE_POOL+1)).clip(0,1).astype('float32')
    return pairs

train_top,train_score,train_target_score=reference_top_candidates(aux_train,REFERENCE_POOL)
valid_top,valid_score,valid_target_score=reference_top_candidates(aux_valid,REFERENCE_POOL)
choice_train_pairs=build_choice_pairs(aux_train,train_top,train_score,train_target_score)
choice_valid_pairs=build_choice_pairs(aux_valid,valid_top,valid_score,valid_target_score)
print(choice_train_pairs.shape,choice_valid_pairs.shape)


## 9. Единый конструктор признаков

`repeat` проверяется по реальной истории данного пользователя. `novelty` — нормированная self-information фильма по train-частоте. Взаимодействия `recent_incomplete × item_light` и `recent_light × item_serious` являются мягкими проверяемыми гипотезами, а не заданными правилами поведения.


In [ ]:
PSYCH_COLUMNS = STATE_COLUMNS + ['novelty','repeat','item_light','item_serious',
    'fatigue_x_light','recent_light_x_serious']
CHOICE_EXTRA = ['gpt_ref_logit_z','gpt_ref_rank_norm'] + PSYCH_COLUMNS
CHOICE_FEATURES = rm.FEATURES + CHOICE_EXTRA
COMPLETION_FEATURES = rm.FEATURES + PSYCH_COLUMNS
CAT_FEATURES = rm.CAT_FEATURES

def enrich_pairs(pairs,contexts,include_choice=True):
    work=pairs.copy()
    # Несколько кандидатов имеют одинаковый context_id, поэтому восстанавливаем порядок через отдельный row id.
    work=work.reset_index(drop=True); work['pair_row_id']=np.arange(len(work),dtype=np.int64)
    base_input=work[['pair_row_id','user_id','item_id','event_day','cutoff','y']].rename(columns={'pair_row_id':'row_id'})
    store.con.register('_candidate_pairs',base_input)
    try: base=store.con.execute(store.feature_sql('SELECT * FROM _candidate_pairs')).df().sort_values('row_id')
    finally: store.con.unregister('_candidate_pairs')
    base=base.reset_index(drop=True)
    states=contexts.set_index('context_id')[STATE_COLUMNS]
    state_block=states.loc[work.context_id].reset_index(drop=True)
    for c in STATE_COLUMNS: base[c]=state_block[c].to_numpy(dtype=np.float32)
    idx=work.item_index.to_numpy(dtype=int)
    histories=contexts.set_index('context_id').history
    history_sets={int(cid):set(map(int,h)) for cid,h in histories.items()}
    base['novelty']=novelty_by_index[idx]
    base['repeat']=np.fromiter((int(item in history_sets[int(cid)]) for cid,item in zip(work.context_id,idx)),
                               dtype=np.float32,count=len(work))
    base['item_light']=light_by_index[idx]
    base['item_serious']=serious_by_index[idx]
    base['fatigue_x_light']=base.recent_incomplete*base.item_light
    base['recent_light_x_serious']=base.recent_light*base.item_serious
    if include_choice:
        base['gpt_ref_logit_z']=work.gpt_ref_logit_z.to_numpy(dtype=np.float32)
        base['gpt_ref_rank_norm']=work.gpt_ref_rank_norm.to_numpy(dtype=np.float32)
    for c in CAT_FEATURES: base[c]=base[c].fillna('__unknown__').astype(str)
    numeric=[c for c in (CHOICE_FEATURES if include_choice else COMPLETION_FEATURES) if c not in CAT_FEATURES]
    for c in numeric: base[c]=pd.to_numeric(base[c],errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
    base['context_id']=work.context_id.to_numpy()
    base['item_index']=idx
    if 'label' in work: base['label']=work.label.to_numpy()
    return base

choice_train=enrich_pairs(choice_train_pairs,aux_train,True)
choice_valid=enrich_pairs(choice_valid_pairs,aux_valid,True)
display(choice_train[CHOICE_FEATURES+['label']].head())


## 10. Choice model с нуля

CatBoostClassifier учится отличать фактически выбранный фильм от сложных альтернатив GPT‑2. Для slate используем raw logits choice model и softmax между фильмами. Абсолютные вероятности не интерпретируем как причинные из-за отсутствия impression-логов.


In [ ]:
choice_model=CatBoostClassifier(
    iterations=700,depth=7,learning_rate=.05,loss_function='Logloss',eval_metric='Logloss',
    random_seed=SEED,l2_leaf_reg=5,random_strength=.5,verbose=100,allow_writing_files=False,
    thread_count=4)
choice_train_pool=Pool(choice_train[CHOICE_FEATURES],choice_train.label,cat_features=CAT_FEATURES)
choice_valid_pool=Pool(choice_valid[CHOICE_FEATURES],choice_valid.label,cat_features=CAT_FEATURES)
choice_model.fit(choice_train_pool,eval_set=choice_valid_pool,early_stopping_rounds=80,use_best_model=True)

valid_prob=choice_model.predict_proba(choice_valid[CHOICE_FEATURES])[:,1]
valid_raw=choice_model.predict(choice_valid[CHOICE_FEATURES],prediction_type='RawFormulaVal')
audit=choice_valid[['context_id','label']].copy(); audit['prob']=valid_prob; audit['raw']=valid_raw
top=audit.sort_values(['context_id','raw'],ascending=[True,False]).groupby('context_id').head(1)
choice_metrics={'logloss':float(log_loss(audit.label,audit.prob)),
                'roc_auc':float(roc_auc_score(audit.label,audit.prob)),
                'group_top1_accuracy':float(top.label.mean()),
                'n_train_pairs':len(choice_train),'n_valid_pairs':len(choice_valid)}
dump_json(OUTPUT_DIR/'choice_metrics.json',choice_metrics)
choice_model.save_model(OUTPUT_DIR/'choice_model.cbm')
display(pd.DataFrame([choice_metrics]))


## 11. Completion model с нуля

Эта модель оценивает `E[watched_pct | фильм выбран]`. Она обучается только на реальных просмотрах и не получает метку 0 для непоказанных фильмов.


In [ ]:
def positive_pairs(contexts):
    p=pd.DataFrame({'context_id':contexts.context_id.astype(int),'user_id':contexts.user_id.astype(int),
        'item_index':contexts.target_index.astype(int),'item_id':contexts.target_item_id.astype(int),
        'cutoff':contexts.cutoff,'event_day':contexts.cutoff,'y':contexts.watched.astype(float)})
    return p[np.isfinite(p.y)].reset_index(drop=True)

completion_train=enrich_pairs(positive_pairs(aux_train),aux_train,False)
completion_valid=enrich_pairs(positive_pairs(aux_valid),aux_valid,False)
completion_train['y']=positive_pairs(aux_train).y.to_numpy()[:len(completion_train)]
completion_valid['y']=positive_pairs(aux_valid).y.to_numpy()[:len(completion_valid)]

completion_model=CatBoostRegressor(
    iterations=900,depth=7,learning_rate=.05,loss_function='RMSE',eval_metric='RMSE',
    random_seed=SEED,l2_leaf_reg=7,random_strength=.5,verbose=100,allow_writing_files=False,
    thread_count=4)
completion_model.fit(Pool(completion_train[COMPLETION_FEATURES],completion_train.y,cat_features=CAT_FEATURES),
    eval_set=Pool(completion_valid[COMPLETION_FEATURES],completion_valid.y,cat_features=CAT_FEATURES),
    early_stopping_rounds=100,use_best_model=True)
cp=np.clip(completion_model.predict(completion_valid[COMPLETION_FEATURES]),0,1)
completion_metrics={'MAE':float(mean_absolute_error(completion_valid.y,cp)),
    'RMSE':float(np.sqrt(mean_squared_error(completion_valid.y,cp))),
    'train_mean_MAE':float(mean_absolute_error(completion_valid.y,np.full(len(cp),completion_train.y.mean()))),
    'n_train':len(completion_train),'n_valid':len(completion_valid)}
dump_json(OUTPUT_DIR/'completion_metrics.json',completion_metrics)
completion_model.save_model(OUTPUT_DIR/'completion_model.cbm')
display(pd.DataFrame([completion_metrics]))


## 12. Support/OOD score

Для каждого числового признака запоминается центральный диапазон train `[1%, 99%]` и IQR. Выход за поддержку превращается в риск `[0,1]`; дополнительно учитываются отсутствующая история фильма и метаданных. Это прозрачный консервативный штраф, который сложнее эксплуатировать, чем ещё одну свободно обучаемую нейросеть.


In [ ]:
OOD_COLUMNS=[c for c in COMPLETION_FEATURES if c not in CAT_FEATURES]
ood_stats={}
for c in OOD_COLUMNS:
    x=pd.to_numeric(choice_train[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    q=x.quantile([.01,.25,.5,.75,.99])
    ood_stats[c]={'q01':float(q.loc[.01]),'q25':float(q.loc[.25]),'q50':float(q.loc[.5]),
                  'q75':float(q.loc[.75]),'q99':float(q.loc[.99])}
dump_json(OUTPUT_DIR/'ood_support.json',ood_stats)

def ood_risk(frame):
    risks=[]
    for c,s in ood_stats.items():
        x=pd.to_numeric(frame[c],errors='coerce').to_numpy(dtype=float)
        scale=max(s['q75']-s['q25'],1e-4)
        z=np.maximum(np.maximum((s['q01']-x)/scale,(x-s['q99'])/scale),0)
        z=np.where(np.isfinite(z),z,2.0)
        risks.append(np.clip(z,0,5))
    r=np.mean(np.vstack(risks),axis=0)
    missing=np.maximum(frame['i_missing'].to_numpy(float),frame['metadata_missing'].to_numpy(float))
    return np.clip(1-np.exp(-r)+.25*missing,0,1).astype(np.float32)

display(pd.Series(ood_risk(choice_valid)).describe())


## 13. Замороженная составная reward-функция

Choice и completion models после этого шага не меняются. Компоненты награды сохраняются отдельно: это позволяет увидеть, не улучшил ли агент итоговый score только ценой повторов, потери разнообразия или ухода за пределы данных.


In [ ]:
def score_slates(contexts, item_matrix, ref_logits_matrix, ref_rank_matrix):
    b,k=item_matrix.shape
    rows=[]
    for i in range(b):
        mean=float(np.mean(ref_logits_matrix[i])); std=max(float(np.std(ref_logits_matrix[i])),1e-4)
        ctx=contexts.iloc[i]
        for j in range(k):
            item=int(item_matrix[i,j])
            rows.append({'context_id':int(ctx.context_id),'user_id':int(ctx.user_id),
                'item_index':item,'item_id':int(index_to_item[item]),'cutoff':ctx.cutoff,
                'event_day':ctx.cutoff,'y':np.nan,'gpt_ref_logit_z':(float(ref_logits_matrix[i,j])-mean)/std,
                'gpt_ref_rank_norm':min(float(ref_rank_matrix[i,j])/(REFERENCE_POOL+1),1.0)})
    pairs=pd.DataFrame(rows)
    feat=enrich_pairs(pairs,contexts,True)
    choice_raw=choice_model.predict(feat[CHOICE_FEATURES],prediction_type='RawFormulaVal').reshape(b,k)
    choice_binary=1/(1+np.exp(-np.clip(choice_raw,-30,30)))
    choice_cond=np.exp(choice_raw-choice_raw.max(axis=1,keepdims=True))
    choice_cond/=choice_cond.sum(axis=1,keepdims=True)
    watched=np.clip(completion_model.predict(feat[COMPLETION_FEATURES]),0,1).reshape(b,k)
    novelty=feat.novelty.to_numpy().reshape(b,k)
    repeat=feat.repeat.to_numpy().reshape(b,k)
    ood=ood_risk(feat).reshape(b,k)
    w=REWARD_WEIGHTS
    item_reward=(w['alpha_log_choice']*np.log(np.clip(choice_binary,1e-5,1))+
                 w['beta_watched_pct']*watched+w['gamma_novelty']*novelty-
                 w['lambda_repeat']*repeat-w['mu_ood']*ood)
    diversity=[]
    for slate in item_matrix:
        g=genre_by_index[slate]
        diversity.append(float(np.mean(g[:,None]!=g[None,:])) if len(g)>1 else 0.0)
    diversity=np.asarray(diversity,dtype=np.float32)
    reward=(choice_cond*item_reward).sum(1)+w['delta_diversity']*diversity
    components={'reward':reward,'log_choice':(choice_cond*np.log(np.clip(choice_binary,1e-5,1))).sum(1),
        'watched_pct':(choice_cond*watched).sum(1),'novelty':(choice_cond*novelty).sum(1),
        'repeat':(choice_cond*repeat).sum(1),'ood':(choice_cond*ood).sum(1),'diversity':diversity}
    return reward,components,feat

# Sanity-check: награда конечна и каждый компонент имеет ожидаемую форму.
test_items=valid_top[:8,:SLATE_SIZE]
test_ref=valid_score[:8,:SLATE_SIZE]
test_rank=np.tile(np.arange(1,SLATE_SIZE+1),(8,1))
test_reward,test_components,_=score_slates(aux_valid.iloc[:8].reset_index(drop=True),test_items,test_ref,test_rank)
assert np.isfinite(test_reward).all()
display(pd.DataFrame(test_components).describe())


## 14. Официальный validation и метрики до RL

Используется тот же `validation_examples.npz`, что и в baseline-ноутбуке. Помимо Recall/NDCG считаются:

- Coverage@10;
- доля ранее просмотренных рекомендаций;
- Gini концентрации exposure;
- персональная novelty;
- intra-list diversity;
- OOD и все составляющие reward.


In [ ]:
def contexts_from_npz(path, history_frame, n, seed, cutoff):
    z=np.load(path)
    rng=np.random.default_rng(seed); take=np.arange(len(z['users'])); rng.shuffle(take); take=take[:n]
    hf=history_frame.sort_values(['user_id','event_time','source_row_index'],kind='stable')
    selected_users=set(map(int,z['users'][take]))
    # Материализуем истории только выбранных для аудита пользователей, а не все 130k+ групп.
    selected_history=hf[hf.user_id.isin(selected_users)]
    groups={int(u):g for u,g in selected_history.groupby('user_id',sort=False)}
    records=[]
    for new_id,i in enumerate(tqdm(take,desc='Official contexts',leave=False)):
        user=int(z['users'][i]); length=int(z['lengths'][i])
        hist=z['histories'][i,:length].astype(np.int32)
        prior=groups.get(user,hf.iloc[:0])
        state=state_from_prior(prior,pd.Timestamp(cutoff),hist)
        rec={'context_id':new_id,'user_id':user,'target_index':int(z['targets'][i]),
             'target_item_id':int(index_to_item[int(z['targets'][i])]),'cutoff':pd.Timestamp(cutoff),
             'history':hist,'watched':np.nan}; rec.update(state); records.append(rec)
    return pd.DataFrame(records)

official_valid=contexts_from_npz(DATA_DIR/'validation_examples.npz',train_df,EVAL_USERS,SEED,
                                 boundaries['validation']['start'])

@torch.inference_mode()
def recommend_topk(model,contexts,k=20):
    model.eval(); out=[]; ref_out=[]; ref_rank=[]
    for start in tqdm(range(0,len(contexts),EVAL_BATCH_SIZE),desc='Recommendations',leave=False):
        part=contexts.iloc[start:start+EVAL_BATCH_SIZE]
        ids,mask=collate_context_rows(part.history.tolist()); last=mask.sum(1)-1
        logits=model(input_ids=ids,attention_mask=mask,use_cache=False).logits
        scores=logits[torch.arange(len(part),device=DEVICE),last,:N_ITEMS].float()
        top=torch.topk(scores,k=k,dim=1,sorted=True).indices
        ref=reference_model(input_ids=ids,attention_mask=mask,use_cache=False).logits
        rs=ref[torch.arange(len(part),device=DEVICE),last,:N_ITEMS].float()
        rv=rs.gather(1,top)
        rtop=torch.topk(rs,k=REFERENCE_POOL,dim=1,sorted=True).indices
        ranks=torch.full_like(top,REFERENCE_POOL+1)
        for j in range(top.shape[1]):
            eq=rtop.eq(top[:,j:j+1]); found=eq.any(1)
            ranks[found,j]=eq[found].float().argmax(1).long()+1
        out.append(top.cpu().numpy()); ref_out.append(rv.cpu().numpy()); ref_rank.append(ranks.cpu().numpy())
    return np.vstack(out),np.vstack(ref_out),np.vstack(ref_rank)

def gini_exposure(items):
    counts=np.bincount(items.ravel(),minlength=N_ITEMS).astype(float)
    if counts.sum()==0:return 0.0
    x=np.sort(counts); n=len(x)
    return float((2*np.dot(np.arange(1,n+1),x)/(n*x.sum()))-(n+1)/n)

def evaluate_policy(model,contexts,label):
    top,ref_scores,ref_ranks=recommend_topk(model,contexts,20)
    target=contexts.target_index.to_numpy()
    metrics={'label':label,'n_users':len(contexts)}
    for k in [10,20]:
        hit=top[:,:k].eq(target[:,None]) if hasattr(top,'eq') else (top[:,:k]==target[:,None])
        metrics[f'Recall@{k}']=float(hit.any(1).mean())
        rank=np.where(hit,np.arange(1,k+1),k+1).min(1)
        metrics[f'NDCG@{k}']=float(np.where(rank<=k,1/np.log2(rank+1),0).mean())
    top10=top[:,:10]
    metrics['Coverage@10']=float(len(np.unique(top10))/N_ITEMS)
    metrics['ExposureGini@10']=gini_exposure(top10)
    rewards=[]; comps=defaultdict(list)
    for start in range(0,len(contexts),512):
        part=contexts.iloc[start:start+512].reset_index(drop=True)
        r,c,_=score_slates(part,top10[start:start+512],ref_scores[start:start+512,:10],ref_ranks[start:start+512,:10])
        rewards.extend(r.tolist())
        for name,v in c.items(): comps[name].extend(np.asarray(v).tolist())
    metrics['CompositeReward']=float(np.mean(rewards))
    for name,v in comps.items(): metrics[f'component_{name}']=float(np.mean(v))
    return metrics,top

baseline_metrics,baseline_recs=evaluate_policy(reference_model,official_valid,'before_RL')
display(pd.DataFrame([baseline_metrics]).T)
dump_json(OUTPUT_DIR/'baseline_policy_metrics.json',baseline_metrics)


## 15. Консервативное дообучение GPT‑2

Используется Plackett–Luce sampling без повторов. Loss состоит из трёх частей:

- policy gradient по замороженной составной reward-функции;
- KL к исходной GPT‑2, ограничивающий distribution shift;
- next-item cross-entropy по настоящему target, защищающий исходную способность угадывать следующий фильм.

Это намеренно консервативнее чистого PPO на синтетической reward model.


In [ ]:
rl_train=aux_train.sample(n=min(RL_TRAIN_CONTEXTS,len(aux_train)),random_state=SEED).reset_index(drop=True)

def plackett_luce_sample(logits,k):
    available=torch.ones_like(logits,dtype=torch.bool)
    chosen=[]; logps=[]
    for _ in range(k):
        masked=logits.masked_fill(~available,-torch.inf)
        dist=torch.distributions.Categorical(logits=masked)
        a=dist.sample(); chosen.append(a); logps.append(dist.log_prob(a))
        available=available.scatter(1,a[:,None],False)
    return torch.stack(chosen,1),torch.stack(logps,1)

optimizer=torch.optim.AdamW(policy_model.parameters(),lr=RL_LEARNING_RATE,weight_decay=.01)
scaler=torch.amp.GradScaler('cuda',enabled=AMP_ENABLED)
history=[]; best_reward=baseline_metrics['CompositeReward']; best_path=OUTPUT_DIR/'best_policy'

for epoch in range(1,RL_EPOCHS+1):
    policy_model.train(); order=np.random.default_rng(SEED+epoch).permutation(len(rl_train))
    for step,start in enumerate(tqdm(range(0,len(order),RL_BATCH_SIZE),desc=f'RL epoch {epoch}')):
        idx=order[start:start+RL_BATCH_SIZE]; part=rl_train.iloc[idx].reset_index(drop=True)
        ids,mask=collate_context_rows(part.history.tolist()); last=mask.sum(1)-1
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda',enabled=AMP_ENABLED):
            logits=policy_model(input_ids=ids,attention_mask=mask,use_cache=False).logits
            current=logits[torch.arange(len(part),device=DEVICE),last,:N_ITEMS].float()
            pool=torch.topk(current,k=POLICY_POOL,dim=1,sorted=True)
            local,logp=plackett_luce_sample(pool.values/RL_TEMPERATURE,SLATE_SIZE)
            chosen=pool.indices.gather(1,local)
            with torch.no_grad():
                ref=reference_model(input_ids=ids,attention_mask=mask,use_cache=False).logits
                ref_full=ref[torch.arange(len(part),device=DEVICE),last,:N_ITEMS].float()
                ref_values=ref_full.gather(1,chosen)
                ref_top=torch.topk(ref_full,k=REFERENCE_POOL,dim=1,sorted=True).indices
                ranks=torch.full_like(chosen,REFERENCE_POOL+1)
                for j in range(SLATE_SIZE):
                    eq=ref_top.eq(chosen[:,j:j+1]); found=eq.any(1)
                    ranks[found,j]=eq[found].float().argmax(1).long()+1
            reward_np,components,_=score_slates(part,chosen.detach().cpu().numpy(),
                ref_values.detach().cpu().numpy(),ranks.detach().cpu().numpy())
            reward=torch.as_tensor(reward_np,device=DEVICE,dtype=torch.float32)
            advantage=(reward-reward.mean())/(reward.std(unbiased=False)+1e-6)
            pg_loss=-(advantage.detach()*logp.sum(1)).mean()
            cur_pool_logp=F.log_softmax(pool.values,dim=1)
            ref_pool=ref_full.gather(1,pool.indices)
            cur_pool_p=cur_pool_logp.exp()
            kl=(cur_pool_p*(cur_pool_logp-F.log_softmax(ref_pool,dim=1))).sum(1).mean()
            target=torch.as_tensor(part.target_index.to_numpy(),device=DEVICE)
            bc=F.cross_entropy(current,target)
            loss=pg_loss+KL_COEF*kl+BC_COEF*bc
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(policy_model.parameters(),MAX_GRAD_NORM)
        scaler.step(optimizer); scaler.update()
        history.append({'epoch':epoch,'step':step,'loss':float(loss.detach()),'pg_loss':float(pg_loss.detach()),
            'kl':float(kl.detach()),'bc':float(bc.detach()),'reward':float(reward.mean()),
            **{f'batch_{k}':float(np.mean(v)) for k,v in components.items()}})

    epoch_metrics,_=evaluate_policy(policy_model,official_valid,f'epoch_{epoch}')
    display(pd.DataFrame([baseline_metrics,epoch_metrics]))
    ndcg_guard=epoch_metrics['NDCG@10'] >= baseline_metrics['NDCG@10']*(1-ALLOWED_RELATIVE_NDCG_DROP)
    if ndcg_guard and epoch_metrics['CompositeReward']>best_reward:
        best_reward=epoch_metrics['CompositeReward']
        policy_model.save_pretrained(best_path,safe_serialization=True)
        print('Сохранён новый лучший checkpoint:',best_path)
    elif not ndcg_guard:
        print('Checkpoint отклонён guardrail: слишком большое падение NDCG@10')

pd.DataFrame(history).to_csv(OUTPUT_DIR/'rl_training_trace.csv',index=False)
if not best_path.exists():
    print('RL не прошёл guardrail/не улучшил reward; сохраняем исходную policy как безопасный результат.')
    reference_model.save_pretrained(best_path,safe_serialization=True)


## 16. Итоговое сравнение и компонентный аудит

Улучшение принимается только вместе с таблицей компонентов. Особое внимание:

- вырос ли `CompositeReward`;
- не рухнули ли Recall/NDCG;
- не выросли ли repeat, OOD и Exposure Gini;
- не получено ли улучшение только одним компонентом при деградации остальных.


In [ ]:
best_model=GPT2LMHeadModel.from_pretrained(best_path,local_files_only=True,attn_implementation='eager').to(DEVICE).eval()
final_metrics,final_recs=evaluate_policy(best_model,official_valid,'after_RL_best')
comparison=pd.DataFrame([baseline_metrics,final_metrics])
numeric=[c for c in comparison.columns if c not in {'label'}]
delta={f'delta_{c}':float(final_metrics[c]-baseline_metrics[c]) for c in numeric if c in final_metrics}
display(comparison)
display(pd.DataFrame([delta]).T)
comparison.to_csv(OUTPUT_DIR/'validation_comparison.csv',index=False)
dump_json(OUTPUT_DIR/'final_policy_metrics.json',final_metrics)
dump_json(OUTPUT_DIR/'metric_deltas.json',delta)

trace=pd.DataFrame(history)
if len(trace):
    fig,ax=plt.subplots(1,3,figsize=(15,4))
    ax[0].plot(trace.reward); ax[0].set_title('Batch composite reward')
    ax[1].plot(trace.kl); ax[1].set_title('KL to reference')
    ax[2].plot(trace.bc); ax[2].set_title('Next-item CE')
    for a in ax:a.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(OUTPUT_DIR/'training_diagnostics.png',dpi=160); plt.show()


## 17. Опциональный финальный test

По умолчанию закрыт. Открывайте его один раз после фиксации весов, коэффициентов reward и всех решений по validation.


In [ ]:
if OPEN_TEST:
    history_for_test=pd.concat([interactions['train'],interactions['validation']],ignore_index=True)
    official_test=contexts_from_npz(DATA_DIR/'test_examples.npz',history_for_test,EVAL_USERS,SEED+100,
                                    boundaries['test']['start'])
    test_before,_=evaluate_policy(reference_model,official_test,'test_before_RL')
    test_after,_=evaluate_policy(best_model,official_test,'test_after_RL')
    pd.DataFrame([test_before,test_after]).to_csv(OUTPUT_DIR/'test_comparison.csv',index=False)
    display(pd.DataFrame([test_before,test_after]))
else:
    print('Test закрыт. Установите OPEN_TEST=True только после фиксации эксперимента.')


## 18. Манифест и архив результатов

Сохраняются исходные хэши, модели, mapping, параметры reward, аудит доступных сигналов и все метрики. Feature-store не включается в архив: он воспроизводимо строится из `prepared.zip`.


In [ ]:
shutil.copy2(DATA_DIR/'item_mapping.json',OUTPUT_DIR/'item_mapping.json')
shutil.copy2(DATA_DIR/'split_boundaries.json',OUTPUT_DIR/'split_boundaries.json')
manifest={
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'run_name':RUN_NAME,'seed':SEED,'device':str(DEVICE),
    'data_hashes':{n:sha256(DATA_DIR/n) for n in REQUIRED},
    'checkpoint_path':str(CHECKPOINT_PATH),
    'reward_weights':REWARD_WEIGHTS,
    'training':{'aux_train_contexts':len(aux_train),'aux_valid_contexts':len(aux_valid),
                'rl_contexts':len(rl_train),'rl_epochs':RL_EPOCHS,'kl_coef':KL_COEF,'bc_coef':BC_COEF},
    'context_signal_audit':context_audit,
    'limitations':[
        'No impression logs: pseudo-negatives are not observed rejections.',
        'Completion is observed only conditional on the historical choice policy.',
        'Calendar-day timestamps cannot identify time of day or sessions.',
        'Offline composite reward is a model-based estimate, not an online causal effect.'
    ]
}
dump_json(OUTPUT_DIR/'manifest.json',manifest)
archive=shutil.make_archive(str(OUTPUT_DIR),'zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name)
print('Результаты:',OUTPUT_DIR)
print('Архив:',archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass
